In [1]:
import pathlib

import apoc
import numpy as np
import pandas as pd
import pyclesperanto_prototype as cle
from bioio import BioImage
from bioio.writers import OmeTiffWriter
from napari_ndev import ImageOverview, helpers
from nyxus import Nyxus
from skimage.morphology import skeletonize

In [2]:
def voronoi_label_adjustment(intensity_image, label_image):
    label_binary = cle.greater_constant(label_image, constant=0) # binarize
    # ferritin_md = cle.median_sphere(intensity_image, radius_x=1, radius_y=1)
    # ferritin_th = cle.top_hat_sphere(ferritin_md, radius_x=8, radius_y=8)
    intensity_th_gb = cle.gaussian_blur(intensity_image, sigma_x=1, sigma_y=1)
    intensity_peaks = cle.detect_maxima_box(intensity_th_gb, radius_x=0, radius_y=0) # this is correct
    select_peaks = cle.binary_and(intensity_peaks, label_binary)
    label_voronoi = cle.masked_voronoi_labeling(select_peaks, label_binary)
    return label_voronoi

def get_pixel_class_as_objects(label, obj_class):
    obj_label = label == obj_class
    return cle.connected_components_labeling_box(obj_label)

def close_labels(label, closing_radius):
    morph_closed = cle.closing_labels(label, radius=closing_radius)
    return cle.connected_components_labeling_box(morph_closed)

def connect_breaks(label, label_connect_distance):
    label_dilated = cle.dilate_labels(label, radius=label_connect_distance/2)
    label_merged = cle.merge_touching_labels(label_dilated)
    label_connected = (label_merged * (label > 0)).astype(np.uint16)
    return label_connected

def exclude_labels(label, minimum_label_size, maximum_label_size):
    label_exclude_large = cle.exclude_labels_on_edges(label)
    label_exclude_small = cle.exclude_labels_out_of_size_range(label_exclude_large, minimum_size=minimum_label_size, maximum_size=maximum_label_size)
    return label_exclude_small

def skeletonize_labels(label):
    skeleton = skeletonize(cle.pull(label))
    skeleton_label = (label * skeleton).astype(np.uint16)
    return skeleton_label

In [3]:
working_dir = pathlib.Path('.')

raw_image_dir, raw_image_files = helpers.get_directory_and_files(
    working_dir / 'Raw_Images'
)

for index, file in enumerate(raw_image_files):
    print(f'{index} : {file.name}')

output_dir = working_dir / 'Processed_Labels_3umconnect'
output_dir.mkdir(exist_ok=True)

overview_dir = output_dir / 'Overviews'
data_dir = working_dir / 'Data'
data_dir.mkdir(exist_ok=True)
data_filename = 'area_intensity_3.csv'

ncoa4_cl = apoc.ObjectSegmenter(working_dir / 'newCD7_classifiers/ncoa4_1.cl')
ftn_cl = apoc.ObjectSegmenter(working_dir / 'newCD7_classifiers/ftn_1.cl')
morph_cl = apoc.PixelClassifier(working_dir / 'newCD7_classifiers/morph_6.cl')
dapi_class_cl = apoc.ObjectClassifier(working_dir / 'newCD7_classifiers/dapi_3_obj_class.cl')

img = BioImage(raw_image_files[0])
pixelsize = img.physical_pixel_sizes.X
print(img.channel_names)
print(img.physical_pixel_sizes)

0 : 2024-08-07 25x 18HIC VERSION 1 NCOA4 647 FT 568 PHALL 488 DAPI OBL.czi
1 : 2024-08-07 25x 24HIC NCOA4 647 FT 568 PHALL 488 DAPI OBL.czi
2 : 2024-08-07 25x 48HIC NCOA4 647 FT 568 PHALL 488 DAPI OBL.czi
3 : 2024-08-07 25x 72HIC NCOA4 647 FT 568 PHALL 488 DAPI OBL.czi
4 : 2024-08-07 25x 96HIC NCOA4 647 FT 568 PHALL 488 DAPI OBL.czi
['AF647', 'AF568', 'AF488', 'DAPI', 'Oblique']
PhysicalPixelSizes(Z=None, Y=0.12408040410185801, X=0.12408040410185801)


In [4]:
DAPI_C = 3
FTN_C = 1
NCOA4_C = 0
PHALL_C = 2

# CONSTANTS, currently as real units, not rounded
NUCLEI_VORONOI_SPOT_SIZE = 3
NUCLEI_VORONOI_OUTLINE_SIZE = 0.3
MIN_NUCLEUS_RADIUS = 3
MAX_NUCLEUS_RADIUS = 10

# Morphology stuff
INITIAL_CLOSING_DISTANCE = 0.1
LABEL_CONNECT_DISTANCE = 3

MAX_PROTEIN_RADIUS = 8

nuclei_voronoi_spot_size = np.round(NUCLEI_VORONOI_SPOT_SIZE / pixelsize)
nuclei_voronoi_outline_size = np.round(NUCLEI_VORONOI_OUTLINE_SIZE / pixelsize)
initial_closing_distance = np.round(INITIAL_CLOSING_DISTANCE/ pixelsize)
label_merge_distance = np.round(LABEL_CONNECT_DISTANCE / pixelsize)
min_nucleus_area = (MIN_NUCLEUS_RADIUS / pixelsize) ** 2 * np.pi
max_nucleus_area = (MAX_NUCLEUS_RADIUS / pixelsize) ** 2 * np.pi
max_protein_area = (MAX_PROTEIN_RADIUS / pixelsize) ** 2 * np.pi

print(f'Nuclei Voronoi Spot Size, px: {nuclei_voronoi_spot_size}')
print(f'Nuclei Voronoi Outline Size, px: {nuclei_voronoi_outline_size}')
print(f'Initial Closing Distance, px: {initial_closing_distance}')
print(f'Label Merge Distance, px: {label_merge_distance}')
print(f'Min Nucleus Area, px^2: {min_nucleus_area}')
print(f'Max Nucleus Area, px^2: {max_nucleus_area}')
print(f'Max Protein Area, px^2: {max_protein_area}')


Nuclei Voronoi Spot Size, px: 24.0
Nuclei Voronoi Outline Size, px: 2.0
Initial Closing Distance, px: 1.0
Label Merge Distance, px: 24.0
Min Nucleus Area, px^2: 1836.4790724572424
Max Nucleus Area, px^2: 20405.323027302697
Max Protein Area, px^2: 13059.40673747372


In [5]:
area_intensity_stack = []

for file in raw_image_files[:]:
    print(file.name)
    img = BioImage(file)
    pixel_size = img.physical_pixel_sizes.X

    for idx, scene in enumerate(img.scenes[:], start=0):
        scene_id = f'{idx}_{img.current_scene_index}_{scene}'
        print(f'{idx} : {img.current_scene_index} :: {scene}')
        img.set_scene(idx)

        dapi = img.get_image_data("YX", C=DAPI_C)
        ncoa4 = img.get_image_data("YX", C=NCOA4_C)
        ftn = img.get_image_data("YX", C=FTN_C)
        phall = img.get_image_data("YX", C=PHALL_C)

        dapi_segmented = cle.voronoi_otsu_labeling(
            dapi, None,
            nuclei_voronoi_spot_size,
            nuclei_voronoi_outline_size
        )
        # dapi_no_edges = cle.exclude_labels_on_edges(dapi_segmented)
        # dapi_exclude = cle.exclude_labels_out_of_size_range(
        #     dapi_no_edges, None, min_nucleus_area, max_nucleus_area
        # )

        dapi_exclude = exclude_labels(dapi_segmented, min_nucleus_area, max_nucleus_area)
        dapi_final_label = dapi_exclude

        # if dapi_exclude.max() > 10:
        #     print("Too many nuclei, skipping")
        #     continue

        dapi_class = dapi_class_cl.predict(labels=dapi_final_label, image=dapi)

        dapi_centroids = cle.reduce_labels_to_centroids(dapi_final_label)
        dapi_class_centroids = ((dapi_centroids > 0) * dapi_class).astype(np.uint16)

        # cle.set_wait_for_kernel_finish(True)
        ftn_seg = ftn_cl.predict(ftn)
        ftn_exclude = exclude_labels(ftn_seg, 0, max_protein_area)
        # ftn_no_edges = cle.exclude_labels_on_edges(ftn_seg)
        # ftn_exclude_large = cle.exclude_labels_out_of_size_range(
        #     ftn_no_edges, None, 0, max_protein_area
        # )
        ferritin_voronoi = voronoi_label_adjustment(ftn, ftn_exclude)
        ferritin_final_label = ferritin_voronoi


        ncoa4_seg = ncoa4_cl.predict(ncoa4)
        # ncoa4_no_edges = cle.exclude_labels_on_edges(ncoa4_seg)
        # ncoa4_exclude_large = cle.exclude_labels_out_of_size_range(
        #     ncoa4_no_edges, None, 0, max_protein_area
        # )
        ncoa4_exclude = exclude_labels(ncoa4_seg, 0, max_protein_area)
        ncoa4_voronoi = voronoi_label_adjustment(ncoa4, ncoa4_exclude)
        ncoa4_final_label = ncoa4_voronoi

        morph_seg = morph_cl.predict([ftn, phall])
        morph_obj = get_pixel_class_as_objects(morph_seg, 2)
        morph_closed = close_labels(morph_obj, initial_closing_distance)
        morph_closed_connected = connect_breaks(morph_closed, label_merge_distance)
        morph_exclude = exclude_labels(morph_closed_connected, min_nucleus_area, 10e10000)
        morph_final = morph_exclude
        morph_skeleton = skeletonize_labels(morph_final)


        ### Save images
        ############################
        image_dict = {
            'image': [dapi, ftn, ncoa4, phall],
            'title': ['DAPI', 'Ferritin', 'NCOA4', 'Phalloidin'],
            'min_display_intensity': [
                np.percentile(dapi, 0.1),
                np.percentile(ftn, 0.1),
                np.percentile(ncoa4, 0.1),
                np.percentile(phall, 0.1),
            ],
            'max_display_intensity': [
                np.percentile(dapi, 99.98),
                np.percentile(ftn, 99.98),
                np.percentile(ncoa4, 99.99),
                np.percentile(phall, 99.98),
            ],
        }


        concatenated_labels = np.stack(
            [
                dapi_final_label,
                ferritin_final_label,
                ncoa4_final_label,
                morph_closed_connected,
                dapi_class,
            ],
            axis=0
        )

        label_names = ["DAPI", "Ferritin", "NCOA4", "Morphology", "DAPI Class"]

        label_dict = {
            'image': concatenated_labels,
            'title': label_names,
            'labels': [True, True, True, True, True],
        }

        ImageOverview(
            [image_dict, label_dict], 8, 8,
            image_title=f"{file.stem}_{scene}",
        ).save(overview_dir, f"{file.stem}_{scene_id}_overview.png")

        OmeTiffWriter.save(
            data=concatenated_labels.astype(np.uint16),
            uri=output_dir / f"{file.stem}_{scene_id}.ome.tiff",
            physical_pixel_sizes=img.physical_pixel_sizes,
            channel_names=label_names,
            dim_order="CYX",
        )

        ############################
        ### MEASUREMENTS
        ############################
        nyxus_dict = {
            # 'description': [
                # intensity_image, intensity_name,
                # label_image, label_name,
            # ],
            'ftn_area_on_morph': [
                ferritin_final_label > 0, 'ferritin_binary',
                morph_final, 'morph_label',
            ],
            'ncoa4_area_on_morph': [
                ncoa4_final_label > 0, 'ncoa4_binary',
                morph_final, 'morph_label',
            ],
            'ftn_intensity_on_morph': [
                ftn, 'ferritin_intensity',
                morph_final, 'morph_label',
            ],
            'ncoa4_intensity_on_morph': [
                ncoa4, 'ncoa4_intensity',
                morph_final, 'morph_label',
            ],
            'ftn_intensity_on_ftn_label': [
                ftn, 'ferritin_intensity',
                ferritin_final_label, 'ferritin_label',
            ],
            'ncoa4_intensity_on_ncoa4_label': [
                ncoa4, 'ncoa4_intensity',
                ncoa4_final_label, 'ncoa4_label',
            ],
            'morph_label_on_ftn_label': [
                morph_final, 'morph_label',
                ferritin_final_label, 'ferritin_label',
            ],
            'morph_label_on_ncoa4_label': [
                morph_final, 'morph_label',
                ncoa4_final_label, 'ncoa4_label',
            ],
            'dapi_class_on_morph': [
                dapi_class_centroids, 'dapi_class',
                morph_final, 'morph_label',
            ],
            'number_and_class_of_nuclei': [
                dapi_class_centroids, 'dapi_class',
                dapi_final_label, 'dapi_label',
            ]
        }

        # Extract values and combine into lists
        nyxus_intensity_list = [value[0] for value in nyxus_dict.values()]
        intensity_names = [value[1] for value in nyxus_dict.values()]
        nyxus_label_list = [value[2] for value in nyxus_dict.values()]
        label_names = [value[3] for value in nyxus_dict.values()]

        area_intensity_nyx = Nyxus(
            features=[
                'area_um2', 'mean', 'median', 'integrated_intensity', 'max',
            ],
            pixels_per_micron=img.physical_pixel_sizes.X,
        )

        area_intensity_df = area_intensity_nyx.featurize(
            intensity_images=np.stack(nyxus_intensity_list, axis=0),
            label_images=np.stack(nyxus_label_list, axis=0),
            intensity_names=intensity_names,
            label_names=label_names,
        )

        # Step 4: Create a mapping fro3 intensity_name and label_name to nyxus_dict key

        # Step 5: Add a new column to 3he DataFrame
        area_intensity_df.insert(0, 'file', file.stem)
        area_intensity_df.insert(1, 'scene', scene)
        area_intensity_df.insert(1, 'scene_id', scene_id)

        mapping = {(value[1], value[3]): key for key, value in nyxus_dict.items()}
        area_intensity_df.insert(3, 'description', area_intensity_df.apply(
            lambda row: mapping.get((row['intensity_image'], row['mask_image'])), axis=1
        ))

        area_intensity_stack.append(area_intensity_df)


area_intensity_concat = pd.concat(area_intensity_stack, ignore_index=True)
# area_intensity_concat.to_csv(data_dir / data_filename, index=False)

2024-08-07 25x 18HIC VERSION 1 NCOA4 647 FT 568 PHALL 488 DAPI OBL.czi
0 : 0 :: P1-A9
1 : 0 :: P4-A9
2 : 1 :: P2-A9
3 : 2 :: P3-A9
4 : 3 :: P4-A9
5 : 1 :: P2-A9
6 : 2 :: P5-A9
7 : 6 :: P3-A9
8 : 3 :: P5-A9
9 : 6 :: P6-A9
10 : 9 :: P6-A9
11 : 9 :: P8-A9
12 : 11 :: P1-A9
13 : 0 :: P7-A9
14 : 13 :: P13-B9
15 : 14 :: P1-B9
16 : 15 :: P3-B9
17 : 16 :: P2-B9
18 : 17 :: P7-B9
19 : 18 :: P6-B9
20 : 19 :: P5-B9
21 : 20 :: P4-B9
22 : 21 :: P8-B9
23 : 22 :: P12-B9
24 : 23 :: P11-B9
25 : 24 :: P9-B9
26 : 25 :: P10-B9
27 : 26 :: P14-B9
28 : 27 :: P4-C9
29 : 28 :: P6-C9
30 : 29 :: P5-C9
31 : 30 :: P2-C9
32 : 31 :: P1-C9
33 : 32 :: P7-C9
34 : 33 :: P12-C9
35 : 34 :: P13-C9
36 : 35 :: P9-C9
37 : 36 :: P8-C9
38 : 37 :: P11-C9
39 : 38 :: P10-C9
40 : 39 :: P14-C9
41 : 40 :: P3-C9
42 : 41 :: P5-D9
43 : 42 :: P3-D9
44 : 43 :: P4-D9
45 : 44 :: P1-D9
46 : 45 :: P6-D9
47 : 46 :: P7-D9
48 : 47 :: P2-D9
49 : 48 :: P8-D9
50 : 49 :: P14-D9
51 : 50 :: P11-D9
52 : 51 :: P9-D9
53 : 52 :: P10-D9
54 : 53 :: P13-D9
55 